In [87]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astroquery.mast import Catalogs, Observations
import astropy.table as at
from astropy import table


### Data Load

In [3]:
data_folder = '/Users/philvanlane/Documents/lc_ae/data/'
hosts = pd.read_csv(data_folder + 'TESS_M_exo_hosts.csv')

### One time formattings to be consistent with code

In [5]:
tic = hosts['tid'].values[0]

In [7]:

obs = Observations.query_criteria(
    target_name=tic, 
    obs_collection="TESS",
    dataproduct_type="timeseries",
    t_exptime=[120,120]
)


In [19]:
record = obs['obs_collection','obs_id','t_exptime','target_name','s_ra','s_dec','sequence_number']
record['tic_searched'] = tic

In [21]:
catalog_data = Catalogs.query_criteria(catalog="Tic", objID=tic)

In [29]:
tic

119081096

In [31]:
catalog_data['objID']

119081096


In [32]:
record['TIC_catalog_id'] = catalog_data['objID']

In [25]:
if len(catalog_data) == 1:
    record['gaia_dr2_id'] = catalog_data['GAIA'][0]
elif len(catalog_data) > 1:
    record['gaia_dr2_id'] = 'multiple'
elif len(catalog_data) == 0:
    record['gaia_dr2_id'] = 'none'

In [26]:
record

obs_collection,obs_id,t_exptime,target_name,s_ra,s_dec,sequence_number,tic_searched,gaia_dr2_id
str4,str47,float64,str9,float64,float64,int64,int64,str19
TESS,tess2021039152502-s0035-0000000119081096-0205-s,120.0,119081096,136.238482652044,-18.502504731515,35,119081096,3380929608950194304
TESS,tess2018206190142-s0001-s0036-0000000119081096,120.0,119081096,136.238482652044,-18.502504731515,36,119081096,3380929608950194304
TESS,tess2019032160000-s0008-0000000119081096-0136-s,120.0,119081096,136.238482652044,-18.502504731515,8,119081096,3380929608950194304


#### For all TICs

In [ ]:
df = pd.DataFrame(columns=record.colnames)

In [48]:
df.drop(df.index, inplace=True)

In [41]:
record = record.to_pandas()

In [42]:
df = pd.concat([df, record], axis=0)

/var/folders/lf/sq3cxxf17kv47p5lcv6w5t5r0000gn/T/ipykernel_89846/2594406524.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, record], axis=0)


In [55]:
tics = hosts['tid'].values

for tic in tics:
    obs = Observations.query_criteria(
        target_name=tic, 
        obs_collection="TESS",
        dataproduct_type="timeseries",
        t_exptime=[120,120]
    )
    
    if len(obs) > 0:
        record = obs['obs_collection','obs_id','t_exptime','target_name','s_ra','s_dec','sequence_number']
        record = record.to_pandas()
    else:
        record = pd.DataFrame(columns=['obs_collection','obs_id','t_exptime','target_name','s_ra','s_dec','sequence_number'])

    record['tic_searched'] = tic

    catalog_data = Catalogs.query_criteria(catalog="Tic", objID=tic)
    if len(catalog_data) > 0:
        record['TIC_catalog_id'] = catalog_data['objID'][0]
    
    if len(catalog_data) == 1:
        record['gaia_dr2_id'] = catalog_data['GAIA'][0]
    elif len(catalog_data) > 1:
        record['gaia_dr2_id'] = 'multiple'
    elif len(catalog_data) == 0:
        record['gaia_dr2_id'] = 'none'

    df = pd.concat([df, record], axis=0)

/var/folders/lf/sq3cxxf17kv47p5lcv6w5t5r0000gn/T/ipykernel_89846/87874877.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, record], axis=0)
/var/folders/lf/sq3cxxf17kv47p5lcv6w5t5r0000gn/T/ipykernel_89846/87874877.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, record], axis=0)
/var/folders/lf/sq3cxxf17kv47p5lcv6w5t5r0000gn/T/ipykernel_89846/87874877.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is depr

In [57]:
df.to_csv(data_folder + 'TESS_M_exo_hosts_obs_summary.csv', index=False)

## Other Data Sources

In [85]:
from astropy.table import Table
t = Table.read(
    data_folder + "TIME_table.mrt",
    format="ascii.cds",
    data_start=30
)

ValueError: Column Prot failed to convert: could not convert string to float: '10  0.'

In [74]:
test = t.to_pandas()

In [75]:
test

,Title: The TIME Table: Rotation and Ages of Cool Exoplanet Host Stars
0,"Authors: Gaidos, E., Claytor, Z., Dungee, R., ..."
1,Table: Catalog of Cool Host Stars with Establi...
2,==============================================...
3,Byte-by-byte Description of file: Table.mrt
4,----------------------------------------------...
...,...
277,KOI-5162 Y 3697 -0.30 17.35 1.24 P ...
278,KOI-2453 Y 3462 -0.42 59.17 5.71 P ...
279,KOI-463 Y 3395 -0.12 50.81 3.65 P ...
280,KOI-249 Y 3547 -0.14 46.43 3.32 P ...


In [62]:
new_df

NameError: name 'new_df' is not defined